# Summary

This notebook shows the performance of different models on the monkey 3-armed bandit task.

---
# Setup

**Installing Gymansium**

To install the gymnasium environment, run `pip install "gymnasium[box2d]"` (or check the gymnasium documentation ??). Might also need to run `brew install swig` before installing gymnasium.

In [ ]:
# @title imports
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from skopt import gp_minimize, dummy_minimize
from skopt.space import Real
from skopt.utils import use_named_args
from skopt.plots import plot_convergence, plot_objective, plot_evaluations


from skopt import gp_minimize, dummy_minimize
from skopt.space import Real
from skopt.utils import use_named_args
from skopt.plots import plot_convergence, plot_objective, plot_evaluations

from popy.simulation_tools import *
from popy.io_tools import *
from popy.behavior_data_tools import *
from popy.plotting.plotting_tools import *

from popy.simulation_helpers import *



ImportError: attempted relative import with no known parent package

In [ ]:
# @title plotting
def _plot_behav(behavior, ax=None):
    # plot the behavior
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    ax.plot(behavior["best_arm"], label="Best arm", color="red", linestyle="--")

    for i, row in behavior.iterrows():
        ax.scatter(row.name, row["action"], color="black", alpha=0.5, marker="o" if row["reward"] == 1 else "x")       
            
    ax.scatter([], [], color="black", alpha=0.5, marker="o", label="Rewarded")
    ax.scatter([], [], color="black", alpha=0.5, marker="x", label="Unrewarded")

    ax.set_xlabel("Time step (trial)")
    ax.set_ylabel("action")
    # add legend: black dot for correct, black x for incorrect, red dashed line for best arm
    
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_yticks([0, 1, 2])
    ax.grid(axis="y")
    ax.spines["top"].set_visible(False)
    
    if ax is None:
        plt.show()
    else:
        return ax
    
def plot_bayesian_bandit(behav):
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    prop_best_arm = (behav["best_arm"].values == behav["action"].values).mean()
    mean_rr = behav["reward"].mean()

    fig.suptitle(f"Bayes agent, \np(best arm): {prop_best_arm:.2f}, rew.rate: {mean_rr:.2f}")
    behav_to_plot = behav.tail(100)
    ax = _plot_behav(behav_to_plot, ax)

    q_values = np.stack(behav_to_plot["posterior"].values)
    ax_ = ax.twinx()
    ax_.plot(behav_to_plot.index+1, q_values[:, 0], label="Posterior state 0", color="blue", linestyle="-", alpha=0.5)  
    ax_.plot(behav_to_plot.index+1, q_values[:, 1], label="Posterior state 1", color="green", linestyle="-", alpha=0.5)
    ax_.plot(behav_to_plot.index+1, q_values[:, 2], label="Posterior state 2", color="orange", linestyle="-", alpha=0.5)
    ax_.set_ylabel("Posterior")
    ax_.legend(bbox_to_anchor=(1.05, 0.75), loc='center left')
    ax_.set_ylim(-.1, 1.1)


    plt.show()
    
def plot_q_agent(behav, title=None):
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    prop_best_arm = (behav["best_arm"].values == behav["action"].values).mean()
    mean_rr = behav["reward"].mean()

    if title is not None:
        fig.suptitle(title + f"\np(best arm): {prop_best_arm:.2f}, rew.rate: {mean_rr:.2f}")
    else:
        fig.suptitle(f"Q-learning agent, \np(best arm): {prop_best_arm:.2f}, rew.rate: {mean_rr:.2f}")
    behav_to_plot = behav.tail(100)
    ax = _plot_behav(behav_to_plot, ax)

    q_values = np.stack(behav_to_plot["q_values"].values)
    ax_ = ax.twinx()
    ax_.plot(behav_to_plot.index+1, q_values[:, 0], label="Q-value arm 0", color="blue", linestyle="-", alpha=0.5)  
    ax_.plot(behav_to_plot.index+1, q_values[:, 1], label="Q-value arm 1", color="green", linestyle="-", alpha=0.5)
    ax_.plot(behav_to_plot.index+1, q_values[:, 2], label="Q-value arm 2", color="orange", linestyle="-", alpha=0.5)
    ax_.set_ylabel("Q-value")
    ax_.legend(bbox_to_anchor=(1.05, 0.7), loc='center left')
    ax_.set_ylim(-1, 1)

    plt.show()

def plot_strategy_value_agent(behav, title=None):
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    prop_best_arm = (behav["best_arm"].values == behav["action"].values).mean()
    mean_rr = behav["reward"].mean()

    if title is not None:
        fig.suptitle(title + f"\np(best arm): {prop_best_arm:.2f}, rew.rate: {mean_rr:.2f}")
    else:
        fig.suptitle(f"Shift-value agent, \np(best arm): {prop_best_arm:.2f}, rew.rate: {mean_rr:.2f}")
    behav_to_plot = behav.tail(100)
    ax = _plot_behav(behav_to_plot, ax)
    ax.grid()

    expectations = np.stack(behav_to_plot["V"].values)
    ax_ = ax.twinx()
    ax_.plot(behav_to_plot.index+1, expectations, label="V", color="blue", linestyle="-", alpha=0.5)  
    ax_.plot(behav_to_plot.index+1, np.ones_like(expectations) * behav_to_plot["V0"].values[0], label="V0", color="green", linestyle="--", alpha=0.5)
    ax_.set_ylabel("V")
    ax_.legend(bbox_to_anchor=(1.05, 0.75), loc='center left')
    ax_.set_ylim(-1, 1)

    plt.show()

def plot_monkey_agent(behav_original, monkey_name):
    behav = behav_original.copy()
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    prop_best_arm = (behav["best_arm"].values == behav["action"].values).mean()
    mean_rr = behav["reward"].mean()

    fig.suptitle(f"Monkey {monkey_name.upper()}, \np(best arm): {prop_best_arm:.2f}, rew.rate: {mean_rr:.2f}")
    behav_to_plot = behav.tail(100)
    ax = _plot_behav(behav_to_plot, ax)
    ax.plot(behav_to_plot["best_arm"], label="Best arm", color="red", linestyle="--")

    plt.show()

def plot_wsls_agent(behav_original):
    behav = behav_original.copy()
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    prop_best_arm = (behav["best_arm"].values == behav["action"].values).mean()
    mean_rr = behav["reward"].mean()

    fig.suptitle(f"WSLS agent, \np(best arm): {prop_best_arm:.2f}, rew.rate: {mean_rr:.2f}")
    behav_to_plot = behav.tail(100)
    ax = _plot_behav(behav_to_plot, ax)
    ax.plot(behav_to_plot["best_arm"], label="Best arm", color="red", linestyle="--")

    plt.show()

def plot_parameter_heatmap(results):
    assert len(results.columns) == 3, "The results should have exactly 3 columns"

    pivot_table = results.pivot(index=results.columns[0], columns=results.columns[1], values='prop_best_arm')
    # round prop_best_arm to 2 decimal places
    pivot_table = pivot_table.round(2)

    # Plot the heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_table, annot=True, fmt="g", cmap="viridis", cbar_kws={'label': 'Proportion of best arm choices'})

    # xticks and yticks rounded to 2 decimal places
    plt.xticks(np.arange(len(results[results.columns[1]].unique())) + 0.5, np.round(results[results.columns[1]].unique(), 2))
    plt.yticks(np.arange(len(results[results.columns[0]].unique())) + 0.5, np.round(results[results.columns[0]].unique(), 2))

    plt.title('Heatmap of the proportion of best arm choices')
    plt.xlabel(results.columns[1])
    plt.ylabel(results.columns[0])
    plt.show()


## Set Environment

In [ ]:
# Create the environment
env = gym.make("zsombi/monkey-bandit-task-v0", n_arms=3, max_episode_steps=10_000_000)

prot_individual_results = False
behavs = {}
best_params = {}

## Load and process monkey data

In [ ]:
behav_monkey = load_behavior()
behav_monkey = drop_time_fields(behav_monkey)
behav_monkey = convert_column_format(behav_monkey, original='behavior')

behav_monkey_ka = behav_monkey[behav_monkey["monkey"] == "ka"]
behav_monkey_po = behav_monkey[behav_monkey["monkey"] == "po"]

'''behav_yuri = load_behavior_yuri()
behav_yuri = convert_column_format(behav_yuri, original='behavior')
behav_yuri_sham = behav_yuri[behav_yuri["monkey"] == "yu_sham"]
behav_yuri_dcz = behav_yuri[behav_yuri["monkey"] == "yu_DCZ"]'''

'behav_yuri = load_behavior_yuri()\nbehav_yuri = convert_column_format(behav_yuri, original=\'behavior\')\nbehav_yuri_sham = behav_yuri[behav_yuri["monkey"] == "yu_sham"]\nbehav_yuri_dcz = behav_yuri[behav_yuri["monkey"] == "yu_DCZ"]'

In [ ]:
# print monkey results
print((behav_monkey_ka["best_arm"].values == behav_monkey_ka["action"].values).mean())
print((behav_monkey_po["best_arm"].values == behav_monkey_po["action"].values).mean())

0.7742568470273881
0.6373878299600445


---
# WSLS Agent



## Advanced WSLS model

In [ ]:
# Define parameter space for ShiftValueAgent
agent_class_wsls = WSLSAgent_custom
model_name = 'WSLS agent'

fixed_params_wsls = {}

# fit the agent
'''param_space = [
    Real(0.0, .3, name='epsilon'),
]

res_temp = fit_agent(agent_class_wsls, param_space, env, 
                    fixed_params=fixed_params_wsls,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_calls=40, n_initial_points=10)

best_params_wsls = res_temp['best_params']
'''

# get best params from previous run
best_params_wsls = {'epsilon': 0.0} 

# simulate the agent
behavior_wsls = simulate_agent(agent_class_wsls, best_params_wsls, env, fixed_params=fixed_params_wsls, behavioral_variables=[])

behavs[model_name] = behavior_wsls
best_params[model_name] = best_params_wsls

print(f'Best parameters: \n{best_params_wsls}')
print('proba best arm: ', (behavior_wsls["best_arm"].values == behavior_wsls["action"].values).mean())

Best parameters: 
{'epsilon': 0.0}
proba best arm:  0.7713306


---
# Section 1. A Q-learning Agent

## Section 1.1.A Standard RL

### Section 1.1.1 No stickyness

In [ ]:
# Define parameter space for ShiftValueAgent
agent_class_qlearn = QLearner
model_name = 'Standard RL'

fixed_params_qlearn = {
    'structure_aware': False
}

# fit the agent
'''
param_space = [
    Real(.01, .9, name='alpha'),
    Real(1, 50, name='beta'),
]
res_temp = fit_agent(agent_class_qlearn, param_space, env, 
                    fixed_params=fixed_params_qlearn,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_jobs=11,
                    n_calls=1100, n_initial_points=1000)

best_params = res_temp['best_params']
'''
# get best params from previous run
best_params_qlearn = {'alpha': 0.8149496594188124, 'beta': 42.18484579352481}

# simulate the agent
behavior_qlearn = simulate_agent(agent_class_qlearn, best_params_qlearn, env, fixed_params_qlearn, behavioral_variables=['q_values'])

behavs[model_name] = behavior_qlearn
best_params[model_name] = best_params_qlearn

print(f'Best parameters: \n{best_params_qlearn}')
print('proba best arm: ', (behavior_qlearn["best_arm"].values == behavior_qlearn["action"].values).mean())

Best parameters: 
{'alpha': 0.8149496594188124, 'beta': 42.18484579352481}
proba best arm:  0.7251991


### Section 1.1.2 With stickyness

In [ ]:
# Define parameter space for ShiftValueAgent
agent_class_qlearn_qlearn_sticky = QLearner
model_name = 'Standard RL - stickyness'

fixed_params_qlearn_sticky = {
    'structure_aware': False
}

# fit the agent
'''
param_space_qlearn_sticky = [
    Real(.01, .9, name='alpha'),
    Real(1, 100, name='beta'),
    Real(0, 50, name='stickyness_bias')
]

res_temp_qlearn_sticky = fit_agent(agent_class_qlearn_qlearn_sticky, param_space_qlearn_sticky, env, 
                    fixed_params=fixed_params_qlearn_sticky,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_jobs=11,
                    n_calls=50, n_initial_points=20)

best_params_qlearn_sticky = res_temp_qlearn_sticky['best_params']
'''
# get best params from previous run
best_params_qlearn_sticky = {'alpha': 0.9, 'beta': 100, 'stickyness_bias': 0.0, 'b1': 0, 'b2': 0, 'b3': 0}  # dont want stickiness

# simulate the agent
behavior_qlearn_sticky = simulate_agent(agent_class_qlearn_qlearn_sticky, best_params_qlearn_sticky, env, fixed_params_qlearn_sticky, behavioral_variables=['q_values'])

behavs[model_name] = behavior_qlearn_sticky
best_params[model_name] = best_params_qlearn_sticky

print(f'Best parameters: \n{best_params_qlearn_sticky}')
print('proba best arm: ', (behavior_qlearn_sticky["best_arm"].values == behavior_qlearn_sticky["action"].values).mean())

Best parameters: 
{'alpha': 0.9, 'beta': 100, 'stickyness_bias': 0.0, 'b1': 0, 'b2': 0, 'b3': 0}
proba best arm:  0.721426


### Section 1.1.3 With forgetting

In [ ]:
# Define parameter space for ShiftValueAgent
agent_class_qlearn_forget = QLearner
model_name = 'Standard RL - forgetting'

fixed_params_qlearn_forget = {
    'structure_aware': False,
    'beta': 100, 
}

# fit the agent
'''
param_space_qlearn_forget = [
    Real(.01, .9, name='alpha'),
    #Real(1, 100, name='beta'),
    Real(0, 1, name='forgetting_rate'),
    Real(0, 1, name='forgetting_threshold')
]

res_temp_qlearn_forget = fit_agent(agent_class_qlearn_forget, param_space_qlearn_forget, env, 
                    fixed_params=fixed_params_qlearn_forget,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_jobs=11,
                    n_calls=70, n_initial_points=20)

best_params_qlearn_forget = res_temp_qlearn_forget['best_params']
'''
# get best params from previous run
best_params_qlearn_forget = {'alpha': 0.332541125297226, 'forgetting_rate': 0.45298043462485366, 'forgetting_threshold': 0.28505530514857486}

# simulate the agent
behavior_qlearn_forget = simulate_agent(agent_class_qlearn_forget, best_params_qlearn_forget, env, fixed_params_qlearn_forget, behavioral_variables=['q_values'])

behavs[model_name] = behavior_qlearn_forget
best_params[model_name] = best_params_qlearn_forget

print(f'Best parameters: \n{best_params_qlearn_forget}')
print('proba best arm: ', (behavior_qlearn_forget["best_arm"].values == behavior_qlearn_forget["action"].values).mean())

Best parameters: 
{'alpha': 0.332541125297226, 'forgetting_rate': 0.45298043462485366, 'forgetting_threshold': 0.28505530514857486}
proba best arm:  0.8050429


### Section 1.1.4 With forgetting + stickyness

In [ ]:
# Define parameter space for ShiftValueAgent
agent_class_qlearn_forget_sticky = QLearner
model_name = 'Standard RL - forgetting + stickyness'

fixed_params_qlearn_forget_sticky = {
    'structure_aware': False,
    'beta': 100, 
    #'forgetting_rate': 0.1,
    #'forgetting_threshold': .1,
}

# fit the agent
'''
param_space_qlearn_forget_sticky = [
    Real(.01, .9, name='alpha'),
    #Real(1, 100, name='beta'),
    Real(0, 1, name='forgetting_rate'),
    Real(0, 1, name='forgetting_threshold'),
    Real(0, 100, name='stickyness_bias')
]

res_temp_qlearn_forget_sticky = fit_agent(agent_class_qlearn_forget_sticky, param_space_qlearn_forget_sticky, env, 
                    fixed_params=fixed_params_qlearn_forget_sticky,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_jobs=11,
                    n_calls=150, n_initial_points=50)

best_params_qlearn_forget_sticky = res_temp_qlearn_forget_sticky['best_params']
'''
# get best params from previous run
best_params_qlearn_forget_sticky = {'alpha': 0.33594497345824015, 'forgetting_rate': 0.4014506611837486, 'forgetting_threshold': 0.2916153937273672, 'stickyness_bias': 0.0}

# simulate the agent
behavior_qlearn_forget_sticky = simulate_agent(agent_class_qlearn_forget_sticky, best_params_qlearn_forget_sticky, env, fixed_params_qlearn_forget_sticky, behavioral_variables=['q_values'])

behavs[model_name] = behavior_qlearn_forget_sticky
best_params[model_name] = best_params_qlearn_forget_sticky

print(f'Best parameters: \n{best_params_qlearn_forget_sticky}')
print('proba best arm: ', (behavior_qlearn_forget_sticky["best_arm"].values == behavior_qlearn_forget_sticky["action"].values).mean())

Best parameters: 
{'alpha': 0.33594497345824015, 'forgetting_rate': 0.4014506611837486, 'forgetting_threshold': 0.2916153937273672, 'stickyness_bias': 0.0}
proba best arm:  0.8027614


## Section 1.2. Inferential RL

### Section 1.2.1 No stickyness

In [ ]:
'''agent_class = QLearner

params = {
    'alpha': 0.1040,
    'beta': 60,  # rate of exploration (i.e. random actions)
}

fixed_params = {
    'structure_aware': True
}

# simulate the agent
behavior_qlearn_inferential = simulate_agent(agent_class, params, env, fixed_params, behavioral_variables=['q_values'])

'''

# Define parameter space for ShiftValueAgent
agent_class_qlearn_inferential = QLearner
model_name = 'Inferential RL'

fixed_params_qlearn_inferential = {
    'structure_aware': True,
    'beta': 100,
}

# fit the agent
'''

param_space_qlearn_inferential = [
    Real(.05, .2, name='alpha'),
    #Real(10, 100, name='beta'),
]

res_temp = fit_agent(agent_class_qlearn_inferential, param_space_qlearn_inferential, env, 
                    fixed_params=fixed_params_qlearn_inferential,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_calls=50, n_initial_points=10)

best_params_qlearn_inferential = res_temp['best_params']
'''
# get best params from previous run
best_params_qlearn_inferential = {'alpha': 0.13, 'beta': 100}
# simulate the agent
behavior_qlearn_inferential = simulate_agent(agent_class_qlearn_inferential, best_params_qlearn_inferential, env, fixed_params_qlearn_inferential, behavioral_variables=['q_values'])

behavs[model_name] = behavior_qlearn_inferential
best_params[model_name] = best_params_qlearn_inferential

print(f'Best parameters: \n{best_params_qlearn_inferential}')
print('proba best arm: ', (behavior_qlearn_inferential["best_arm"].values == behavior_qlearn_inferential["action"].values).mean())

Best parameters: 
{'alpha': 0.13, 'beta': 100}
proba best arm:  0.8044299


### Section 1.2.2 With stickyness

In [ ]:
'''agent_class = QLearner

params = {
    'alpha': 0.1040,
    'beta': 60,  # rate of exploration (i.e. random actions)
}

fixed_params = {
    'structure_aware': True
}

# simulate the agent
behavior_qlearn_inferential = simulate_agent(agent_class, params, env, fixed_params, behavioral_variables=['q_values'])

'''

# Define parameter space for ShiftValueAgent
agent_class_qlearn_inferential_sticky = QLearner
model_name = 'Inferential RL - stickyness'

fixed_params_qlearn_inferential_sticky = {
    'structure_aware': True,
}

# fit the agent
'''
param_space_sticky = [
    #Real(.05, .2, name='alpha'),
    #Real(10, 100, name='beta'),
    Real(0.0, 50, name='stickyness')
]

res_temp_sticky = fit_agent(agent_class_qlearn_inferential_sticky, param_space_sticky, env, 
                    fixed_params=fixed_params_qlearn_inferential_sticky,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_calls=150, n_initial_points=100)

best_params_qlearn_inferential_sticky = res_temp_sticky['best_params']
'''

# get best params from previous run
best_params_qlearn_inferential_sticky = {'alpha': 0.2, 'beta': 100.81579533811015, 'stickyness_bias': 25} #best is .2, 100, 25

# simulate the agent
behavior_qlearn_inferential_sticky = simulate_agent(agent_class_qlearn_inferential_sticky, best_params_qlearn_inferential_sticky, env, fixed_params_qlearn_inferential_sticky, behavioral_variables=['q_values'])

behavs[model_name] = behavior_qlearn_inferential_sticky
best_params[model_name] = best_params_qlearn_inferential_sticky

print(f'Best parameters: \n{best_params_qlearn_inferential_sticky}')
print('proba best arm: ', (behavior_qlearn_inferential_sticky["action"].values == behavior_qlearn_inferential_sticky["best_arm"].values).mean())

Best parameters: 
{'alpha': 0.2, 'beta': 100.81579533811015, 'stickyness_bias': 25}
proba best arm:  0.8093326


### Optimal parameters

In [ ]:
"""# grid search
alpha_range = np.linspace(.01, 1, 10)
beta = 100
stickyness_range = np.linspace(0.0, 50.0, 10)

res_grid_qlearn_inferential = np.zeros((len(alpha_range), len(stickyness_range)))

for i, alpha in enumerate(alpha_range):
    for j, stickyness in enumerate(stickyness_range):
        params_temp = {
            'alpha': alpha,
            'beta': beta,
            'stickyness_bias': stickyness
        }
        behavior_temp = simulate_agent(QLearner, params_temp, env, fixed_params_qlearn_inferential_sticky, behavioral_variables=['q_values'])

        prop_best_arm = (behavior_temp["best_arm"].values == behavior_temp["action"].values).mean()

        res_grid_qlearn_inferential[i, j] = prop_best_arm * 100

# save the results (numpy array)
#floc = 'results/inferential_rl_grid_search.npy'
#np.save(floc, res_grid_qlearn_inferential)

'''# load the results
floc = 'results/inferential_rl_grid_search.npy'
res_grid = np.load(floc)
res_grid = res_grid * 100'''"""

'# grid search\nalpha_range = np.linspace(.01, 1, 10)\nbeta = 100\nstickyness_range = np.linspace(0.0, 50.0, 10)\n\nres_grid_qlearn_inferential = np.zeros((len(alpha_range), len(stickyness_range)))\n\nfor i, alpha in enumerate(alpha_range):\n    for j, stickyness in enumerate(stickyness_range):\n        params_temp = {\n            \'alpha\': alpha,\n            \'beta\': beta,\n            \'stickyness_bias\': stickyness\n        }\n        behavior_temp = simulate_agent(QLearner, params_temp, env, fixed_params_qlearn_inferential_sticky, behavioral_variables=[\'q_values\'])\n\n        prop_best_arm = (behavior_temp["best_arm"].values == behavior_temp["action"].values).mean()\n\n        res_grid_qlearn_inferential[i, j] = prop_best_arm * 100\n\n# save the results (numpy array)\n#floc = \'results/inferential_rl_grid_search.npy\'\n#np.save(floc, res_grid_qlearn_inferential)\n\n\'\'\'# load the results\nfloc = \'results/inferential_rl_grid_search.npy\'\nres_grid = np.load(floc)\nres_grid 

In [ ]:
"""import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.transforms import Bbox
import numpy as np

# Set font sizes
plt.rcParams.update({'font.size': 8})

# Create a larger figure to accommodate the axis labels and colorbar
fig, ax = plt.subplots(1, 1, figsize=(4/2.54, 4/2.54))  # Slightly larger to fit everything

# Create colors from red to blue
colors = plt.cm.cividis(np.linspace(0, 1, len(stickyness_range)))

for j, stickyness in enumerate(stickyness_range):
    # Use your chopped data
    ax.plot(alpha_range, res_grid_qlearn_inferential[:, j], color=colors[j])

# Add parameter points with compact styling
ka_params = cfg.MODEL_PARAMS_RL['ka']
po_params = cfg.MODEL_PARAMS_RL['po']
yu_sham_params = cfg.MODEL_PARAMS_RL['yu_sham']
yu_DCZ_params = cfg.MODEL_PARAMS_RL['yu_DCZ']

ax.scatter(ka_params['alpha'], np.interp(ka_params['alpha'], alpha_range, res_grid_qlearn_inferential[:, 0]),
           color=COLORS['ka'], s=70, label='KA', 
          edgecolor='white', linewidth=1, zorder=10)
ax.scatter(po_params['alpha'], np.interp(po_params['alpha'], alpha_range, res_grid_qlearn_inferential[:, 0]), color=COLORS['po'], s=70, label='PO', 
          edgecolor='white', linewidth=1, zorder=10)
'''ax.scatter(yu_sham_params['alpha'], np.interp(yu_sham_params['alpha'], alpha_range, res_grid_qlearn_inferential), color=COLORS['yu_sham'], s=50, label='YU_sham',
          edgecolor='white', linewidth=1, zorder=10)
ax.scatter(yu_DCZ_params['alpha'], np.interp(yu_DCZ_params['alpha'], alpha_range, res_grid_qlearn_inferential), color    =COLORS['yu_DCZ'], s=40, label='YU_DCZ',
          edgecolor='white', linewidth=1, zorder=10)'''

# Labels
ax.set_xlabel('Learning Rate (α)')
ax.set_ylabel('% optimal target selection')

ax.set_xlim(0, .5)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add colorbar outside the main plot
'''cbar = fig.colorbar(im, ax=ax, fraction=0.044, pad=.05)#, label='Proportion of best target selection')
cbar.ax.tick_params(labelsize=6)'''

# Add compact legend
ax.legend(frameon=True, loc='lower right', fontsize=6, markerscale=0.8)

# Save with exact dimensions preserved
#plt.savefig('figs/grid_search_inferential_rl.svg', dpi=300, bbox_inches='tight', transparent=True)
plt.show()"""

"import matplotlib.pyplot as plt\nimport matplotlib as mpl\nfrom matplotlib.transforms import Bbox\nimport numpy as np\n\n# Set font sizes\nplt.rcParams.update({'font.size': 8})\n\n# Create a larger figure to accommodate the axis labels and colorbar\nfig, ax = plt.subplots(1, 1, figsize=(4/2.54, 4/2.54))  # Slightly larger to fit everything\n\n# Create colors from red to blue\ncolors = plt.cm.cividis(np.linspace(0, 1, len(stickyness_range)))\n\nfor j, stickyness in enumerate(stickyness_range):\n    # Use your chopped data\n    ax.plot(alpha_range, res_grid_qlearn_inferential[:, j], color=colors[j])\n\n# Add parameter points with compact styling\nka_params = cfg.MODEL_PARAMS_RL['ka']\npo_params = cfg.MODEL_PARAMS_RL['po']\nyu_sham_params = cfg.MODEL_PARAMS_RL['yu_sham']\nyu_DCZ_params = cfg.MODEL_PARAMS_RL['yu_DCZ']\n\nax.scatter(ka_params['alpha'], np.interp(ka_params['alpha'], alpha_range, res_grid_qlearn_inferential[:, 0]),\n           color=COLORS['ka'], s=70, label='KA', \n        

### Section 1.2.3 Multiple alphas

In [ ]:
# Define parameter space for ShiftValueAgent
agent_class_qlearn_inferential_multialpha = QLearner
model_name = 'Inferential RL - stickyness + multiple alphas'

fixed_params_qlearn_inferential_multialpha = {
    'structure_aware': True,
    'beta': 100,
}

# fit the agent

param_space_multialpha = [
    Real(0.01, .9, name='alpha'),
    Real(0, .9, name='alpha_unchosen'),
    Real(0.0, 15, name='stickyness_bias')
]

res_temp_multialpha = fit_agent(agent_class_qlearn_inferential_multialpha, param_space_multialpha, env, 
                    fixed_params=fixed_params_qlearn_inferential_multialpha,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_calls=200, n_initial_points=150)

best_params_qlearn_inferential_multialpha = res_temp_multialpha['best_params']
'''

# get best params from previous run
best_params_qlearn_inferential_multialpha = {'alpha': 0.1791262718549457, 'alpha_unchosen': 0.08933117915871931} #best is .2, 100, 25
'''
# simulate the agent
behavior_qlearn_inferential_multialpha = simulate_agent(agent_class_qlearn_inferential_multialpha, best_params_qlearn_inferential_multialpha, env, fixed_params_qlearn_inferential_multialpha, behavioral_variables=['q_values'])

behavs[model_name] = behavior_qlearn_inferential_multialpha
best_params[model_name] = best_params_qlearn_inferential_multialpha

print(f'Best parameters: \n{best_params_qlearn_inferential_multialpha}')
print('proba best arm: ', (behavior_qlearn_inferential_multialpha["action"].values == behavior_qlearn_inferential_multialpha["best_arm"].values).mean())

In [ ]:
# alpha vs alpha unchosen: it really reflects half the information??
...

---
# Section 2. Foraging model


## Section 2.1.1. No reset

In [ ]:
'''agent_class = ShiftValueAgent

params = {
    'alpha': 0.4770,
    'beta': 98.10,
    'V0': 0.1973
}

fixed_params = {
    'reset_on_switch': False
}

# simulate the agent
behavior_foraging = simulate_agent(agent_class, params, env, fixed_params, behavioral_variables=["V", "V0"])'''



# Define parameter space for ShiftValueAgent
agent_class_foraging = ForagingAgent
model_name = 'Foraging - no reset'

fixed_params_foraging = {
    'reset_on_switch': False
}

# fit the agent
'''
param_space_foraging = [
    Real(.1, .8, name='alpha'),
    Real(10, 100, name='beta'),
    Real(0, 1, name='V0'),
]

res_temp_foraging = fit_agent(agent_class_foraging, param_space_foraging, env, 
                    fixed_params=fixed_params_foraging,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_calls=150, n_initial_points=100)

best_params_foraging = res_temp_foraging['best_params']
'''
# get best params from previous run
best_params_foraging = {'alpha': 0.41728277205284503, 'beta': 1000.0, 'V0': 0.21028761393188744}

# simulate the agent
behavior_foraging = simulate_agent(agent_class_foraging, best_params_foraging, env, fixed_params_foraging, behavioral_variables=["V", "V0"])

behavs[model_name] = behavior_foraging
best_params[model_name] = best_params_foraging

print(f'Best parameters: \n{best_params_foraging}')
print('proba best arm: ', (behavior_foraging["best_arm"].values == behavior_foraging["action"].values).mean())

## Section 2.1.2. Reset

In [ ]:
'''agent_class = ShiftValueAgent

params = {
    'alpha': 0.4903,
    'beta': 71.26,
    'V0': 0.2062
}

fixed_params = {
    'reset_on_switch': True
}

# simulate the agent
behavior_foraging_reset = simulate_agent(agent_class, params, env, fixed_params, behavioral_variables=["V", "V0"])
'''

# Define parameter space for ShiftValueAgent
agent_class_foraging_reset = ForagingAgent
model_name = 'Foraging'

fixed_params_foraging_reset = {
    'reset_on_switch': True,
}

# fit the agent

param_space_foraging_reset = [
    Real(.05, .85, name='alpha'),
    Real(10, 100, name='beta'),
    Real(0.1, .5, name='V0'),
]

'''res_temp_foraging_reset = fit_agent(agent_class_foraging_reset, param_space_foraging_reset, env, 
                    fixed_params=fixed_params_foraging_reset,
                    fit_on='rr',
                    make_plots=True, verbose=True,
                    n_calls=200, n_initial_points=50)'''
'''res_temp_foraging_reset = fit_agent_strict(agent_class_foraging_reset, param_space_foraging_reset, env,
                    fixed_params=fixed_params_foraging_reset,
                    fit_on='rr',
                    n_restarts=100,
                    verbose=True)

best_params_foraging_reset = res_temp_foraging_reset['best_params']

'''
# get best params from previous run
best_params_foraging_reset = {'alpha': 0.2925040720063767, 'beta': 100.0, 'V0': 0.30962820339137886}

# simulate the agent
behavior_foraging_reset = simulate_agent(agent_class_foraging_reset, best_params_foraging_reset, env, fixed_params_foraging_reset, behavioral_variables=["V", "V0"])

behavs[model_name] = behavior_foraging_reset
best_params[model_name] = best_params_foraging_reset

print(f'Best parameters: \n{best_params_foraging_reset}')
print('proba best arm: ', (behavior_foraging_reset["best_arm"].values == behavior_foraging_reset["action"].values).mean())

## Section 2.1.3. Abandoned bias

In [ ]:
'''agent_class = ShiftValueAgent

params = {
    'alpha': 0.4903,
    'beta': 71.26,
    'V0': 0.2062
}

fixed_params = {
    'reset_on_switch': True
}

# simulate the agent
behavior_foraging_reset = simulate_agent(agent_class, params, env, fixed_params, behavioral_variables=["V", "V0"])
'''

# Define parameter space for ShiftValueAgent
agent_class_foraging_reset_memory = ForagingAgent
model_name = 'Foraging - abandoned bias'

fixed_params_foraging_reset_memory = {
    'reset_on_switch': True,
    'beta': 100,
    #'spatial_bias': [0, 0, 0],
    'abandoned_bias': -50,
    'abandoned_decay': 1,
}

# fit the agent
'''
param_space_foraging_reset_memory = [
    Real(.05, .85, name='alpha'),
    #Real(10, 100, name='beta'),
    Real(0.1, .5, name='V0'),
    #Real(0, 1, name='abandoned_decay'),
]

res_temp_foraging_reset_memory = fit_agent(agent_class_foraging_reset_memory, param_space_foraging_reset_memory, env,
                    fixed_params=fixed_params_foraging_reset_memory,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_calls=150, n_initial_points=50)
best_params_foraging_reset_memory = res_temp_foraging_reset_memory['best_params']

# get best params from previous run
'''
best_params_foraging_reset_memory = {'alpha': 0.253480491296527, 'beta': 100.0, 'V0': 0.33564298467705633}
#best_params_foraging_reset_memory = {'alpha': 0.3571295025358409, 'V0': 0.23610157896957373, 'abandoned_decay': 1.0}

# simulate the agent
behavior_foraging_reset_memory = simulate_agent(agent_class_foraging_reset_memory, best_params_foraging_reset_memory, env, fixed_params_foraging_reset_memory, behavioral_variables=["V", "V0"])

behavs[model_name] = behavior_foraging_reset_memory
best_params[model_name] = best_params_foraging_reset_memory

print(f'Best parameters: \n{best_params_foraging_reset_memory}')
print('proba best arm: ', (behavior_foraging_reset_memory["best_arm"].values == behavior_foraging_reset_memory["action"].values).mean())

## Optimal parameters

In [ ]:
"""# grid search
alpha_range = np.linspace(.01, 1, 50)
beta = 100
V0_range = np.linspace(0, 1, 50)

'''
res_grid = np.zeros((len(alpha_range), len(V0_range)))

for i, alpha in enumerate(alpha_range):
    for j, V0 in enumerate(V0_range):
        params = {
            'alpha': alpha,
            'beta': beta,
            'V0': V0
        }
        behavior = simulate_agent(agent_class, params, env, fixed_params, behavioral_variables=["V", "V0"])

        prop_best_arm = (behavior["best_arm"].values == behavior["action"].values).mean()

        res_grid[i, j] = prop_best_arm * 100

# save the results (numpy array)
floc = 'results/shift_value_agent_grid_search.npy'
np.save(floc, res_grid)'''

# load the results
floc = 'results/shift_value_agent_grid_search.npy'
res_grid = np.load(floc)
res_grid = res_grid * 100"""

In [ ]:
"""import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.transforms import Bbox
import numpy as np

# Set font sizes
plt.rcParams.update({'font.size': 8})

# Create a larger figure to accommodate the axis labels and colorbar
fig, ax = plt.subplots(1, 1, figsize=(4/2.54, 4/2.54))  # Slightly larger to fit everything

# Create nice colormap for the heatmap
cmap = plt.cm.cividis

# Use your chopped data
res_grid_chopped = res_grid[4:41, 0:36]
alpha_range_chopped = alpha_range[4:41]
V0_range_chopped = V0_range[0:36]

im = ax.imshow(res_grid_chopped, origin='lower', 
               extent=[V0_range_chopped[0], V0_range_chopped[-1], alpha_range_chopped[0], alpha_range_chopped[-1]], 
               aspect='equal', interpolation='gaussian', cmap=cmap)

# Add contour lines at every 0.1 increment with minimal labels
levels = np.arange(0, 100, 10)
contour = ax.contour(
    np.linspace(V0_range_chopped[0], V0_range_chopped[-1], res_grid_chopped.shape[1]),
    np.linspace(alpha_range_chopped[0], alpha_range_chopped[-1], res_grid_chopped.shape[0]), 
    res_grid_chopped, levels=levels, colors='white', linewidths=0.5, alpha=0.7
)
# Only label a few contours to avoid clutter
ax.clabel(contour, inline=True, fontsize=6, fmt='%.0f', levels=levels)

# Add parameter points with compact styling
ka_params = cfg.MODEL_PARAMS['ka']
po_params = cfg.MODEL_PARAMS['po']
yu_sham_params = cfg.MODEL_PARAMS['yu_sham']
yu_DCZ_params = cfg.MODEL_PARAMS['yu_DCZ']


ax.scatter(ka_params['V0'], ka_params['alpha'], color=COLORS['ka'], s=50, label='KA', 
          edgecolor='white', linewidth=1, zorder=10)
ax.scatter(po_params['V0'], po_params['alpha'], color=COLORS['po'], s=50, label='PO', 
          edgecolor='white', linewidth=1, zorder=10)
ax.scatter(yu_sham_params['V0'], yu_sham_params['alpha'], color=COLORS['yu_sham'], s=50, label='YU_sham',
          edgecolor='white', linewidth=1, zorder=10)
'''ax.scatter(yu_DCZ_params['V0'], yu_DCZ_params['alpha'], color=COLORS['yu_DCZ'], s=40, label='YU_DCZ',
          edgecolor='white', linewidth=1, zorder=10)'''

# Labels
ax.set_xlabel('Initial Value (V0)')
ax.set_ylabel('Learning Rate (α)')

# Add colorbar outside the main plot
'''cbar = fig.colorbar(im, ax=ax, fraction=0.044, pad=.05)#, label='Proportion of best target selection')
cbar.ax.tick_params(labelsize=6)'''

# Add compact legend
ax.legend(frameon=True, loc='upper right', fontsize=6, markerscale=0.8)

# Save with exact dimensions preserved
plt.savefig('figs/grid_search_shift_value_reset.svg', dpi=300, bbox_inches='tight', transparent=True)
"""

In [ ]:
if prot_individual_results:
    # plot the behavior
    plot_strategy_value_agent(behavior_foraging_reset, title='Shift-value agent with reset')
    plot_summary_stats(behavior_foraging_reset, title='Shift-value agent with reset')
    plot_hist_thingy(behavior_foraging_reset, title='Shift-value agent with reset')

---
# Section 3. HMM normative Agent

## Section 3.1. Model (MODEL V)

In [ ]:
'''import numpy as np
from scipy.special import logsumexp

class BayesianAgent:
    def __init__(
        self,
        n_arms=3,
        transition_matrix=np.array([[39/40, .5/40, .5/40], 
                                  [.5/40, 39/40, .5/40], 
                                  [.5/40, .5/40, 39/40]]),
        emission_matrix_pos=np.array([[.7, .25, .25], 
                                    [.25, .7, .25], 
                                    [.25, .25, .7]]),
        emission_matrix_neg=np.array([[.3, .75, .75], 
                                    [.75, .3, .75], 
                                    [.75, .75, .3]]),
        beta=100,
        block_length_mean=40,
        block_length_std=5,
        transition_length=5
    ):
        """
        Initialize optimal Bayesian agent for multi-armed bandit with hidden state transitions.
        
        Args:
            n_arms: Number of arms
            transition_matrix: State transition probabilities
            emission_matrix_pos: Reward probabilities for positive outcomes
            emission_matrix_neg: Reward probabilities for negative outcomes  
            beta: Inverse temperature for softmax action selection
            block_length_mean: Mean block length
            block_length_std: Standard deviation of block length
            transition_length: Number of trials with gradual transition
        """
        self.n_arms = n_arms
        self.transition_matrix = transition_matrix
        self.emission_matrix_pos = emission_matrix_pos
        self.emission_matrix_neg = emission_matrix_neg
        self.beta = beta
        
        # Block timing parameters
        self.block_length_mean = block_length_mean
        self.block_length_std = block_length_std
        self.transition_length = transition_length
        
        # State tracking
        self.max_block_length = int(block_length_mean + 3 * block_length_std)
        self.reset()
        self.rewards = []
    
    def reset(self):
        """Reset the agent's beliefs to initial state."""
        # Joint belief over (state, time_in_block, block_length)
        # Shape: (n_states, max_block_length, possible_block_lengths)
        self.min_block_length = max(1, int(self.block_length_mean - 3 * self.block_length_std))
        self.possible_block_lengths = range(self.min_block_length, self.max_block_length + 1)
        
        # Initialize uniform belief over states and block parameters
        self.joint_posterior = np.ones((
            self.n_arms, 
            self.max_block_length, 
            len(self.possible_block_lengths)
        )) / (self.n_arms * len(self.possible_block_lengths))
        
        # Only the first time step is possible initially
        self.joint_posterior[:, 1:, :] = 0
        self.joint_posterior = self.joint_posterior / np.sum(self.joint_posterior)
        
        self.trial_count = 0
    
    def _get_block_length_prior(self, block_length):
        """Get prior probability of a block length (truncated normal)."""
        # Simplified: uniform over reasonable range
        if self.min_block_length <= block_length <= self.max_block_length:
            return 1.0 / len(self.possible_block_lengths)
        return 0.0
    
    def _get_transition_probabilities(self, time_in_block, block_length):
        """
        Get state transition probabilities considering gradual transitions.
        
        During the last transition_length trials, probabilities gradually shift.
        """
        if time_in_block <= block_length - self.transition_length:
            # Before transition period - stable state
            return self.transition_matrix
        elif time_in_block > block_length:
            # After block end - complete transition
            return np.array([[0, 0.5, 0.5], [0.5, 0, 0.5], [0.5, 0.5, 0]])
        else:
            # During transition period - gradual change
            progress = (time_in_block - (block_length - self.transition_length)) / self.transition_length
            stable_trans = self.transition_matrix
            switch_trans = np.array([[0, 0.5, 0.5], [0.5, 0, 0.5], [0.5, 0.5, 0]])
            return (1 - progress) * stable_trans + progress * switch_trans
    
    def _get_emission_probabilities(self, time_in_block, block_length, reward, action):
        """
        Get emission probabilities considering gradual reward transitions.
        """
        if time_in_block <= block_length - self.transition_length:
            # Before transition - stable rewards
            if reward > 0:
                return self.emission_matrix_pos[:, action]
            else:
                return self.emission_matrix_neg[:, action]
        elif time_in_block > block_length:
            # After transition - switched rewards
            switched_pos = np.roll(self.emission_matrix_pos, 1, axis=0)  # Rotate states
            switched_neg = np.roll(self.emission_matrix_neg, 1, axis=0)
            if reward > 0:
                return switched_pos[:, action]
            else:
                return switched_neg[:, action]
        else:
            # During transition - gradual change
            progress = (time_in_block - (block_length - self.transition_length)) / self.transition_length
            if reward > 0:
                stable_emit = self.emission_matrix_pos[:, action]
                switched_emit = np.roll(self.emission_matrix_pos, 1, axis=0)[:, action]
            else:
                stable_emit = self.emission_matrix_neg[:, action]
                switched_emit = np.roll(self.emission_matrix_neg, 1, axis=0)[:, action]
            
            return (1 - progress) * stable_emit + progress * switched_emit
    
    def get_action_probabilities(self):
        """
        Compute optimal action probabilities using expected rewards.
        """
        # Marginalize to get state beliefs
        state_beliefs = np.sum(self.joint_posterior, axis=(1, 2))
        
        # Compute expected reward for each action
        expected_rewards = np.zeros(self.n_arms)
        
        for action in range(self.n_arms):
            # Expected reward = P(reward=1|action) for each state, weighted by state beliefs
            for state in range(self.n_arms):
                # Get current time and block length distributions
                time_block_dist = self.joint_posterior[state, :, :]
                
                for t in range(self.max_block_length):
                    for bl_idx, block_length in enumerate(self.possible_block_lengths):
                        if time_block_dist[t, bl_idx] > 1e-10:
                            # Get emission probability for this time/block combination
                            emit_probs = self._get_emission_probabilities(t, block_length, 1, action)
                            expected_rewards[action] += (time_block_dist[t, bl_idx] * 
                                                       emit_probs[state])
        
        # Apply softmax with temperature
        exp_rewards = np.exp(self.beta * expected_rewards)
        return exp_rewards / np.sum(exp_rewards)
    
    def act(self):
        """Select action using softmax policy."""
        action_probs = self.get_action_probabilities()
        return np.random.choice(self.n_arms, p=action_probs)
    
    def update_values(self, action, reward):
        """
        Update beliefs using Bayesian inference.
        """

        self.rewards.append(reward) 
        if len(self.rewards) % 100 == 0:
            print(np.mean(np.array(self.rewards)))

        self.trial_count += 1
        new_posterior = np.zeros_like(self.joint_posterior)
        
        for state in range(self.n_arms):
            for t in range(self.max_block_length):
                for bl_idx, block_length in enumerate(self.possible_block_lengths):
                    if self.joint_posterior[state, t, bl_idx] < 1e-10:
                        continue
                    
                    # Emission probability
                    emit_prob = self._get_emission_probabilities(t, block_length, reward, action)[state]
                    
                    # Update for next time step
                    if t + 1 < self.max_block_length:
                        # Continue in same block
                        new_posterior[state, t + 1, bl_idx] += (
                            self.joint_posterior[state, t, bl_idx] * emit_prob
                        )
                    
                    # Handle block transitions
                    if t >= block_length:
                        # Transition to new block and new state
                        trans_probs = self._get_transition_probabilities(t, block_length)
                        for new_state in range(self.n_arms):
                            for new_bl_idx, new_block_length in enumerate(self.possible_block_lengths):
                                new_posterior[new_state, 0, new_bl_idx] += (
                                    self.joint_posterior[state, t, bl_idx] * 
                                    emit_prob * 
                                    trans_probs[state, new_state] *
                                    self._get_block_length_prior(new_block_length)
                                )
        
        # Normalize
        total_prob = np.sum(new_posterior)
        if total_prob > 1e-10:
            self.joint_posterior = new_posterior / total_prob
        else:
            # Numerical stability - reset if beliefs become too concentrated
            self.reset()
    
    def get_state_beliefs(self):
        """Get marginal beliefs over states."""
        return np.sum(self.joint_posterior, axis=(1, 2))
    
    def get_expected_block_position(self):
        """Get expected position within current block."""
        weights = np.sum(self.joint_posterior, axis=0)
        if np.sum(weights) > 1e-10:
            positions = np.arange(self.max_block_length)[:, None]
            return np.sum(positions * weights) / np.sum(weights)
        return 0'''

In [ ]:
'''agent_class = BayesianAgent

params = {
    'beta': 100,
}

fixed_params = {
    'transition_matrix': np.array([[39/40, .5/40, .5/40], [.5/40, 39/40, .5/40], [.5/40, .5/40, 39/40]]),
    'emission_matrix_pos': np.array([[.7, .25, .25], [.25, .7, .25], [.25, .25, .7]]),
    'emission_matrix_neg': np.array([[.3, .75, .75], [.75, .3, .75], [.75, .75, .3]]),
}

# simulate the agent
behavior_bayes = simulate_agent(agent_class, params, env, fixed_params, behavioral_variables=['posterior'])
'''
env = gym.make("zsombi/monkey-bandit-task-v0", n_arms=3, max_episode_steps=10_000)

# Define parameter space for ShiftValueAgent
agent_class_bayes = BayesianAgent

p_s = 0.03222797620935691
fixed_params_bayes = {
    #'transition_matrix': np.array([[p_t, p_s, p_s], [p_s, p_t, p_s], [p_s, p_s, p_t]]),
    #'p_switch': p_s,
    #'emission_matrix_pos': np.array([[.7, .25, .25], [.25, .7, .25], [.25, .25, .7]]),
    #'emission_matrix_neg': np.array([[.3, .75, .75], [.75, .3, .75], [.75, .75, .3]]),
}

# fit the agent
'''
param_space_bayes = [
    Real(1/50, 1/30, name='p_switch'),  # inverse of the probability to stay in the same state
]

res_temp_bayes = fit_agent(agent_class_bayes, param_space_bayes, env, 
                    fixed_params=fixed_params_bayes,
                    fit_on='rr',
                    make_plots=True, verbose=False,
                    n_calls=150, n_initial_points=100)

best_params_bayes = res_temp_bayes['best_params']
'''
# get best params from previous run
best_params_bayes = {'beta': 200}

# simulate the agent
behavior_bayes = simulate_agent(agent_class_bayes, best_params_bayes, env, fixed_params_bayes, behavioral_variables=[])

print(f'Best parameters: \n{best_params_bayes}')

print(f'prop best arm: ')
print((behavior_bayes["best_arm"].values == behavior_bayes["action"].values).mean())

---
# Summary

In [ ]:
def plot_performances(behavs, y_label=None, title=None, paper_format=True, savedir=None):
    if paper_format:
        h = 4  # height of the plot in cm
        w = 4  # width of the plot in cm
    else:
        h = 7
        w = 7
    fig, ax = plt.subplots(1, 1, figsize=(w/2.54, h/2.54))
    for key, value in behavs.items():
        prop_best_arm = (value["best_arm"].values == value["action"].values).mean()
        mean_rr = value["reward"].mean()

        if key == 'Bayesian':
            ax.axhline(prop_best_arm * 100, color='black', linestyle='--', label='Normative')
            # print 'Optimal performance' over line
            ax.text(0, prop_best_arm * 100 + 1, 'Optimal performance', color='black', fontsize=8, ha='left', va='bottom')

            print(f'{key}: \np(best arm): {prop_best_arm:.3f}, rew.rate: {mean_rr:.3f}')

        else:

            if key in COLORS:
                color = COLORS[key]
            else:
                color = 'grey'

            ax.bar(key, prop_best_arm * 100 , alpha=0.8, color=color, zorder=2, edgecolor='black', linewidth=0.5)
            print(f'{key}: \np(best arm): {prop_best_arm:.3f}, rew.rate: {mean_rr:.3f}')


    # rotate x-axis labels
    plt.xticks(rotation=45, horizontalalignment='right', fontsize=8)

    #ax.set_ylabel("Proportion of best arm choices")
    #ax.set_title("Performance of different agents")
    ax.set_ylim(50, 85)
    if y_label is not None:
        ax.set_ylabel(y_label)
    if title is not None:
        ax.set_title(title)
    # yticks at every .1
    ax.set_yticks(np.arange(50, 85, 10))

    #ax.grid(axis="y", zorder=0, alpha=.5)
    #ax.grid(axis="y", zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_ylabel("% optimal target selection")

    # save the figure
    return fig, ax

In [ ]:
'''# plot the learning curves and the proba shift together
behavs = {
    'WSLS': behavior_wsls,
    'Standard RL-s': behavior_qlearn,
    'Standard RL': behavior_qlearn_sticky,
    'Inferential RL-s': behavior_qlearn_inferential,
    'Inferential RL': behavior_qlearn_inferential_sticky,
    #'Shift-value': behavior_foraging,
    'Foraging': behavior_foraging_reset,
    'Bayesian': behavior_bayes,
    'Monkey KA': behav_monkey_ka,
    'Monkey PO': behav_monkey_po,
    #'Monkey YU (sham)': behav_yuri_sham,
    #'Monkey YU (DCZ)': behav_yuri_dcz,
}'''

fig, ax = plot_performances(behavs, paper_format=False)

# save as svg
fig.savefig('figs/performances.svg', dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
raise ValueError('stop here')

In [ ]:
behavs_pd = []
for model, behav_temp in behavs.items():
    if model in ['Monkey KA', 'Monkey PO', 'Monkey YU (sham)', 'Monkey YU (DCZ)']:
        behav_temp['monkey'] = behav_temp['monkey']
        behav_temp['model'] = behav_temp['monkey']
    else:
        behav_temp["model"] = model
        behav_temp["monkey"] = model
        behav_temp['session'] = 0
        
    behav_temp = convert_column_format(behav_temp, original='simulation')
    
    behavs_pd.append(behav_temp)    
behavs_pd = pd.concat(behavs_pd)    

fig, axs = plot_strategy(behavs_pd, paper_format=False, verbose=False)


In [ ]:
fig, axs = plot_strategy(behavs_pd, paper_format=True, verbose=False)


## Statistical differences between models

In [ ]:
#rates_wsls, ratesB, ratesC = [], [], []
rates_wsls = []
rates_qlearn = []
rates_foraging = []
rates_qlearn_inferential = []
rates_bayes = []

for i in range(100):
    if i % 10 == 0: print(f'Simulation {i}/100')
    behavior_wsls = simulate_agent(agent_class_wsls, best_params_wsls, env, fixed_params=fixed_params_wsls, behavioral_variables=[])
    behavior_qlearn = simulate_agent(agent_class_qlearn, best_params_qlearn, env, fixed_params_qlearn, behavioral_variables=['q_values'])
    behavior_qlearn_inferential = simulate_agent(agent_class_qlearn_inferential, best_params_qlearn_inferential, env, fixed_params_qlearn_inferential, behavioral_variables=['q_values'])
    behavior_foraging = simulate_agent(agent_class_foraging_reset, best_params_foraging_reset, env, fixed_params_foraging_reset, behavioral_variables=["V", "V0"])
    behavior_bayes = simulate_agent(agent_class_bayes, best_params_bayes, env, fixed_params_bayes, behavioral_variables=['posterior'])

    prop_best_wsls = (behavior_wsls["best_arm"].values == behavior_wsls["action"].values).mean()
    prop_best_qlearn = (behavior_qlearn["best_arm"].values == behavior_qlearn["action"].values).mean()
    prop_best_qlearn_inferential = (behavior_qlearn_inferential["best_arm"].values == behavior_qlearn_inferential["action"].values).mean()
    prop_best_foraging = (behavior_foraging["best_arm"].values == behavior_foraging["action"].values).mean()
    prop_best_bayes = (behavior_bayes["best_arm"].values == behavior_bayes["action"].values).mean()

    rates_wsls.append(prop_best_wsls)
    rates_qlearn.append(prop_best_qlearn)
    rates_qlearn_inferential.append(prop_best_qlearn_inferential)
    rates_foraging.append(prop_best_foraging)
    rates_bayes.append(prop_best_bayes)

rates_wsls = np.array(rates_wsls)
rates_qlearn = np.array(rates_qlearn)
rates_qlearn_inferential = np.array(rates_qlearn_inferential)
rates_foraging = np.array(rates_foraging)
rates_bayes = np.array(rates_bayes)


In [ ]:
# print mean rates per model to 2 decimals
print(f'Foraging: {rates_foraging.mean()*100:.2f} ± {rates_foraging.std()*100:.2f}')
print(f'Inferential RL: {rates_qlearn_inferential.mean()*100:.2f} ± {rates_qlearn_inferential.std()*100:.2f}')
print(f'Standard RL: {rates_qlearn.mean()*100:.2f} ± {rates_qlearn.std()*100:.2f}')
print(f'WSLS: {rates_wsls.mean()*100:.2f} ± {rates_wsls.std()*100:.2f}')
print(f'Bayesian: {rates_bayes.mean()*100:.2f} ± {rates_bayes.std()*100:.2f}')

# do pairwise U tests

from scipy import stats

# wsls vs forage / inferential
u, p = stats.mannwhitneyu(rates_wsls, rates_qlearn_inferential, alternative='two-sided')
print(f"WSLS - RL: Mann–Whitney U: U={u:.0f}, p={p:.3g}, n1={len(rates_wsls)}, n2={len(rates_qlearn_inferential)}")
u, p = stats.mannwhitneyu(rates_wsls, rates_foraging, alternative='two-sided')
print(f"WSLS - Forage: Mann–Whitney U: U={u:.0f}, p={p:.3g}, n1={len(rates_wsls)}, n2={len(rates_foraging)}")

# standard vs inferential / forage
u, p = stats.mannwhitneyu(rates_qlearn, rates_qlearn_inferential, alternative='two-sided')
print(f"RL - Inferential RL: Mann–Whitney U: U={u:.0f}, p={p:.3g}, n1={len(rates_qlearn)}, n2={len(rates_qlearn_inferential)}")
u, p = stats.mannwhitneyu(rates_qlearn, rates_foraging, alternative='two-sided')
print(f"RL - Forage: Mann–Whitney U: U={u:.0f}, p={p:.3g}, n1={len(rates_qlearn)}, n2={len(rates_foraging)}")

# inferential vs forage
u, p = stats.mannwhitneyu(rates_qlearn_inferential, rates_foraging, alternative='two-sided')
print(f"Inferential RL - Forage: Mann–Whitney U: U={u:.0f}, p={p:.3g}, n1={len(rates_qlearn_inferential)}, n2={len(rates_foraging)}")

# Bayesian foraging / inferential
u, p = stats.mannwhitneyu(rates_bayes, rates_qlearn_inferential, alternative='two-sided')
print(f"Bayesian - Inferential RL: Mann–Whitney U: U={u:.0f}, p={p:.3g}, n1={len(rates_bayes)}, n2={len(rates_qlearn_inferential)}")
u, p = stats.mannwhitneyu(rates_bayes, rates_foraging, alternative='two-sided')
print(f"Bayesian - Forage: Mann–Whitney U: U={u:.0f}, p={p:.3g}, n1={len(rates_bayes)}, n2={len(rates_foraging)}")              

# plot distributions
fig, ax = plt.subplots(1, 1, figsize=(4/2.54, 4/2.54))
sns.kdeplot(rates_wsls * 100, label='WSLS', color=COLORS['WSLS'], fill=True, alpha=0.5, linewidth=1, ax=ax)
sns.kdeplot(rates_qlearn * 100, label='Standard RL', color=COLORS['Standard RL'], fill=True, alpha=0.5, linewidth=1, ax=ax)
sns.kdeplot(rates_qlearn_inferential * 100, label='Inferential RL', color=COLORS['Inferential RL'], fill=True, alpha=0.5, linewidth=1, ax=ax)
sns.kdeplot(rates_foraging * 100, label='Foraging', color=COLORS['Foraging'], fill=True, alpha=0.5, linewidth=1, ax=ax)
sns.kdeplot(rates_bayes * 100, label='Bayesian', color=COLORS['Bayesian'], fill=True, alpha=0.5, linewidth=1, ax=ax)

# show means and 95% CI
ax.axvline(rates_wsls.mean() * 100, color=COLORS['WSLS'], linestyle='--', linewidth=1)
ax.axvline(rates_qlearn.mean() * 100, color=COLORS['Standard RL'], linestyle='--', linewidth=1)
ax.axvline(rates_qlearn_inferential.mean() * 100, color=COLORS['Inferential RL'], linestyle='--', linewidth=1)
ax.axvline(rates_foraging.mean() * 100, color=COLORS['Foraging'], linestyle='--', linewidth=1)
ax.axvline(rates_bayes.mean() * 100, color=COLORS['Bayesian'], linestyle='--', linewidth=1) 

ax.set_xlabel('% optimal target selection')
ax.set_ylabel('Density')
ax.legend(frameon=True, loc='upper right', fontsize=6, markerscale=0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# measure confidence intervals with bootstrapping
